```txt
【数据准备阶段】
1. download_dataset.py↓ 下载完整数据集
   mobvoi_seq_monkey_general_open_corpus.jsonl (完整版)
   
2. process_dataset.ipynb
   ↓ 提取前100万行
   mobvoi_seq_monkey_general_open_corpus_small.jsonl (测试版)

【训练阶段】
3. pretrain.sh
   ↓ 指定数据路径参数
   --train_files mobvoi_seq_monkey_general_open_corpus_small.jsonl
   
4. pretrain.py
   ↓ 读取数据文件
   load_dataset('json', data_files=data_args.train_files)
   ↓ 在代码内部进行数据处理
   - tokenize（第178-191行）
   - 文本切块（第209-232行）
```

# pretrain.sh文件 
我来为您详细解释pretrain.sh的作用和各个参数的含义。

## pretrain.sh的核心作用

**pretrain.sh是预训练的启动脚本**，它的主要作用是：

1. **配置训练环境**（设置GPU）
2. **启动分布式训练**（使用DeepSpeed）
3. **传递训练参数**（给pretrain.py）

## 脚本结构分析

### 1. GPU配置（第1行）
```bash
CUDA_VISIBLE_DEVICES=0,1
```
- **作用**：指定使用哪些GPU
- **示例**：`0,1`表示使用0号和1号GPU（双卡训练）
- **可修改**：根据你的GPU数量调整，如`0,1,2,3`（四卡）

### 2. DeepSpeed启动命令（第3行）
```bash
deepspeed pretrain.py \
```
- **作用**：使用DeepSpeed框架启动分布式训练
- **替代**：如果不用DeepSpeed，可以用`python pretrain.py`（单卡）

### 3. 模型配置参数

<thinking>
用户需要了解pretrain.sh的作用，我应该详细解释每个参数的含义。让我分类说明这些参数。
</thinking>

```bash
--config_name autodl-tmp/qwen-1.5b
--tokenizer_name autodl-tmp/qwen-1.5b
```
- **config_name**：模型配置文件路径（从零初始化模型）
- **tokenizer_name**：分词器路径

### 4. 数据配置参数
```bash
--train_files autodl-tmp/dataset/pretrain_data/mobvoi_seq_monkey_general_open_corpus_small.jsonl
--block_size 2048
--preprocessing_num_workers 10
```
- **train_files**：训练数据文件路径
- **block_size**：文本块长度（2048个token）
- **preprocessing_num_workers**：数据预处理线程数

### 5. 训练超参数
```bash
--per_device_train_batch_size 16
--gradient_accumulation_steps 4
--learning_rate 1e-4
--num_train_epochs 1
--warmup_steps 200
```
- **per_device_train_batch_size**：每张GPU的batch size
- **gradient_accumulation_steps**：梯度累积步数
  - 实际batch size = 16 × 4 × 2(GPU数) = 128
- **learning_rate**：学习率
- **num_train_epochs**：训练轮数
- **warmup_steps**：学习率预热步数

### 6. 输出和日志配置
```bash
--output_dir autodl-tmp/output/pretrain
--logging_dir autodl-tmp/output/pretrain/logs
--logging_steps 5
--save_steps 100
--save_total_limit 1
```
- **output_dir**：模型输出目录
- **logging_dir**：日志保存目录
- **logging_steps**：每5步打印一次日志
- **save_steps**：每100步保存一次checkpoint
- **save_total_limit**：最多保留1个checkpoint

### 7. 优化配置
```bash
--bf16
--gradient_checkpointing
--deepspeed ./ds_config_zero2.json
```
- **bf16**：使用bfloat16混合精度训练
- **gradient_checkpointing**：梯度检查点（节省显存）
- **deepspeed**：DeepSpeed配置文件（ZeRO-2优化）

### 8. 监控配置
```bash
--report_to swanlab
```
- **report_to**：使用SwanLab进行训练监控

### 9. 其他配置
```bash
--do_train
--evaluation_strategy no
--seed 12
```
- **do_train**：执行训练
- **evaluation_strategy**：不进行评估
- **seed**：随机种子

## 完整执行流程

```
1. 设置GPU环境变量
   ↓
2. DeepSpeed启动pretrain.py
   ↓
3. pretrain.py接收所有参数
   ↓
4. 加载模型配置和tokenizer
   ↓
5. 加载训练数据
   ↓
6. 初始化Trainer
   ↓
7. 开始分布式训练
   ↓
8. 定期保存checkpoint和日志
   ↓
9. 训练完成，保存最终模型
```

## 如何使用

```bash
# 直接运行
bash pretrain.sh

# 或者给予执行权限后运行
chmod +x pretrain.sh
./pretrain.sh
```

## 常见修改场景

### 修改GPU数量
```bash
# 使用4张GPU
CUDA_VISIBLE_DEVICES=0,1,2,3
```

### 使用完整数据集
```bash
# 将small改为完整文件名
--train_files autodl-tmp/dataset/pretrain_data/mobvoi_seq_monkey_general_open_corpus.jsonl
```

### 调整batch size
```bash
# 减小显存占用
--per_device_train_batch_size 8
--gradient_accumulation_steps 8
# 实际batch size保持不变：8 × 8 × 2 = 128
```

### 从checkpoint恢复训练
```bash
# 取消注释最后一行
--resume_from_checkpoint autodl-tmp/output/pretrain/checkpoint-20400
```

## 总结

**pretrain.sh的本质**：一个配置文件 + 启动命令的组合，它让你无需在代码中硬编码参数，而是通过命令行参数灵活配置训练过程。这是工业级训练的标准做法。

# pretrain 和 finetune

> pretrain 是"读书学语言"，所有 token 都要预测；finetune 是"学对话"，只对 assistant 的回答计算 loss，让模型学会按指令回答问题。

## 1. 模型初始化
pretrain.py 支持两种方式：

config_name →随机初始化（从零训练）
model_name_or_path → 加载已有权重（继续训练）
finetune.py 只有一种：

model_name_or_path → 必须加载预训练权重，没有从零初始化的选项

## 2. 数据格式与处理
pretrain.py 处理的是纯文本：

输入：{"text": "..."} 的 jsonl 文件
处理方式：tokenize 后将所有文本拼接，再按 block_size 切块
labels 直接复制 input_ids（全部 token 都参与 loss 计算）
finetune.py 处理的是对话数据：

输入：{"conversations": [{"from": "human", "value": "..."}, {"from": "assistant", "value": "..."}]} 格式
处理方式：用 preprocess() 函数构建 ChatML 格式，拼接多轮对话
labels 中 system 和 human 部分被设为 IGNORE_TOKEN_ID，只有 assistant 的回复参与 loss 计算

## 3. 数据集类
pretrain.py 直接用 HuggingFace datasets 库的 map 处理。

finetune.py 自定义了 SupervisedDataset(Dataset) 类，封装了对话数据的加载和预处理逻辑。

## 4. Trainer 配置
pretrain.py 使用了 default_data_collator：


trainer = Trainer(..., data_collator=default_data_collator)
finetune.py 没有指定 data_collator（因为 SupervisedDataset 已经处理好了 padding 和 attention_mask）。

## 5. SwanLab 项目名
pretrain：project="pretrain", experiment_name="from_scrach"
finetune：project="sft", experiment_name="qwen-1.5b"

# DeepSpeed ZeRO-2

> 相关参考资料：(ai推荐， 未考证)   
1. 📺 视频讲解 (YouTube / Bilibili)

Microsoft Research 官方介绍 (YouTube)

标题: ZeRO: Memory Optimization Toward Training Trillion Parameter Models

- 亮点: 这是 ZeRO 论文第一作者 Samyam Rajbhandari 亲自讲解的。虽然是 2020 年的视频，但它把为什么会有内存冗余、如何通过“数据并行”的改进来消除冗余讲得最透彻。

Hugging Face 官方技术分享 (YouTube)

- 关键词: Hugging Face DeepSpeed Integration

- 亮点: 如果你想知道 DeepSpeed 是如何跟你的 pretrain.py（基于 Transformers 库）联动的，这个视频是必看。它更偏向工程实践。

李沐 (Mu Li) 的论文精读系列 (Bilibili)

- 亮点: 虽然李沐老师可能没有专门出一期只讲 DeepSpeed 的视频，但在他关于分布式训练和大规模模型的精读视频里，经常会深度剖析 ZeRO 的内存占用公式。

2. ✍️ 顶级技术博客 (必读)
Microsoft DeepSpeed 官方博客 (最权威)

- 推荐文章: DeepSpeed: Extreme-scale model training for everyone

- 核心内容: 里面那张经典的“内存占用对比图”是所有炼丹师的启蒙图。它详细对比了传统数据并行、ZeRO-1、ZeRO-2 和 ZeRO-3 的显存节省比例。

Hugging Face 官方文档 / 博客 (最实用)

- 推荐文章: Efficient Training on Multiple GPUs

- 核心内容: 这篇文章把 ZeRO-2 的原理用非常直观的语言描述了出来：“大家分摊任务，不要每个人都背着全套的优化器状态”。



## 👯‍♂️ 第一部分：GPU 0 和 GPU 1 到底是怎么分工的？ 
在分布式训练中，你的两张显卡既是**“平等的苦力”，又有“阶级之分”**。

1. 阶级之分：GPU 0 是“带头大哥”，GPU 1 是“无情打工仔”  

GPU 0（主卡 / Main Process）：它不仅要干重体力活，还要负责所有的统筹和后勤。

- 发号施令：你在屏幕上看到的那些带有 INFO 的白色日志，全都是 GPU 0 打印出来的。GPU 1 是个哑巴，不允许往屏幕上打印东西（不然两张卡同时打印，屏幕就乱码了）。

- 备菜（我们刚讲过的）：负责把 20GB 的数据切好、翻译好，存进硬盘。

- 存盘（Save Checkpoint）：训练到最后，负责把最终变聪明的模型权重打包存进你的 /output 文件夹里的，也是 GPU 0。

GPU 1（副卡）：它没有任何统筹权限，它只负责一件事——疯狂算数学题。


2. 平等干活：它们怎么协同计算？（数据并行）  

还记得我们之前算的那道数学题吗？你设置了单卡 batch_size = 8。
在每一轮的训练中，两张卡的配合极其默契：

- 分发试卷：总管家给 GPU 0 发 8 句待学习的文本，给 GPU 1 发另外 8 句不同的文本。

- 低头狂算：两张卡同时开始算前向传播，各自算出自己手头这 8 句话的误差（Loss）和梯度（经验值）。

- “对答案”（通信同步）：算完之后，两张卡会通过主板上的数据线（PCIe 甚至 NVLink）瞬间进行一次极其高速的开会：“喂，我这边的梯度是这么多，你那边是多少？咱们加起来平均一下！”（这在底层叫做 All-Reduce）。

- 共同进步：拿到平均后的统一梯度后，两张卡同时更新自己肚子里的模型参数。这样就保证了两张卡里的模型永远是一模一样的。



## 🪄 第二部分：DeepSpeed 到底是个什么神仙？
一句话总结：DeepSpeed 是微软开发的一款“超级外挂”，它是当今世界上能让平民显卡跑起大模型的唯一救星。

如果你不用 DeepSpeed，直接用 PyTorch 原生的多卡训练去跑 Qwen-1.5B，你的两张 3090 会在启动的第 1 秒钟就直接报错 OOM（爆显存） 原地升天。

为什么会爆显存？（大模型的“内存墙”诅咒）
很多人以为 1.5B（15亿）参数的模型很小，也就 3GB 左右。**但训练和推理完全不同！**
在训练时，除了模型本身，显存里还要塞进极其庞大的 **“优化器状态（Optimizer States）”（比如 Adam 算法会记录过去的动量，它的体积是模型参数的 2 到 3 倍）和梯度（Gradients）**。加起来，单张 3090 的 24G 显存根本吃不下。


**DeepSpeed 的终极魔法：ZeRO（零冗余优化器）**  
你看你的 pretrain.sh 最后一行写着 `--deepspeed ./ds_config_zero2.json`。这个 ZeRO-2 就是 DeepSpeed 施展的具体魔法。

传统的训练就像两个士兵（GPU 0 和 GPU 1）每人都要背一个 300 斤的超级大背包（包含完整的模型+所有优化器状态），结果两个人都被压死了。

DeepSpeed ZeRO-2 跑过来说：“你们傻啊！背包里的东西有很多是重复的！”
于是，DeepSpeed 做了极其变态的 **“显存切割”**：

它把那坨最重、最占空间的“优化器状态”和“梯度”用刀劈成了两半。

让 GPU 0 只背上半部分，GPU 1 只背下半部分。

效果：每张显卡的显存压力瞬间降低了 50%！两张 3090 顿时觉得身轻如燕，不但能跑了，还能吃下更长的文本（block_size 2048）。



> 需要先搞清楚训练时显存都在哪？**(大模型训练的内存结构)**     
1️⃣ 模型参数 (weights)  
2️⃣ 梯度 (gradients)  
3️⃣ 优化器状态 (optimizer states)  
4️⃣ 激活值 (activations)  
‼️ 优化器状态  >  梯度  >  参数

> ZeRO-2 的核心思想：  
  参数可以复制      
  但梯度和优化器状态可以分摊

| Stage  | 拆什么              |
| ------ | ---------------- |
| ZeRO-1 | 只拆优化器            |
| ZeRO-2 | 拆优化器 + 梯度        |
| ZeRO-3 | 参数 + 梯度 + 优化器 全拆 |



# CPU & GPU

# 预训练任务
## 预训练一般有什么任务？

在 NLP 里，大模型预训练常见任务主要有三类：



### ① CLM（Causal Language Modeling）

代表模型：

* GPT-2
* GPT-3
* LLaMA

训练目标：

> 用前面的词预测下一个词

公式是：

$$
P(x_t | x_1, x_2, ..., x_{t-1})
$$

例如：

```
今天天气很好，我想去
```

模型预测：

```
公园
```

特点：

* 单向（从左到右）
* 自回归
* 推理时和训练一致

👉 这是目前主流 LLM 的预训练方式



### ② MLM（Masked Language Modeling）

代表模型：

* BERT
* RoBERTa

训练方式：

把一句话中 15% 的词随机遮盖：

```
今天天气[MASK]好
```

模型预测：

```
很
```

特点：

* 双向
* 只能做理解任务
* 不适合直接生成文本



### ③ Seq2Seq 预训练

代表模型：

* T5
* BART

做法：

* 输入被破坏的句子
* 输出原句

例如：

```
输入：今天天气很好 [删除部分]
输出：今天天气很好，我想去公园
```

适合翻译、摘要、问答。



## CLM 到底是什么？

CLM = Causal Language Modeling

Causal 的意思是：

> 当前 token 只能看“过去”，不能看未来。

在 Transformer 里实现方式：

* 使用下三角 mask
* 禁止注意力看到未来 token

Attention 矩阵变成：

```
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

这叫：

**因果掩码（causal mask）**



## CLM 的 loss 是怎么算的？

假设输入：

```
我 爱 深 度 学 习
```

模型训练时：

| 输入    | 预测目标 |
| ----- | ---- |
| 我     | 爱    |
| 我 爱   | 深    |
| 我 爱 深 | 度    |
| ...   | ...  |

本质是：

把 input_ids 向右平移一位作为 labels。

代码通常是：

```python
labels = input_ids.clone()
```

然后内部自动 shift。



## 现在的 pretrain.py 是哪种？

如果你看到代码里有：

```python
AutoModelForCausalLM
```

那就是 CLM。

来自：

Transformers



## 为什么现在主流都是 CLM？

因为：

1. 生成能力强
2. 推理时逻辑和训练一致
3. scaling law 表现最好
4. 适合 instruction tuning

所以：

现代 LLM = 基本都是 CLM 预训练



## 预训练完整流程（以 CLM 为例）

```
原始文本
  ↓
tokenizer
  ↓
拼接成大长序列
  ↓
按 block_size 切块
  ↓
input_ids
  ↓
Causal Mask
  ↓
预测下一个 token
  ↓
CrossEntropyLoss
```





# 高阶函数与callback
> 代码问题1 ：“我明明连括号都没加，连参数都没传，它怎么就自己跑起来了，而且还知道 examples 是什么？”
> 代码问题2: 为什么两个函数里的 examples 绝对不会冲突？

```python 
 def tokenize_function(examples): # examples: {"text": [...], ...}  (原始数据集格式)
    return tokenizer(examples["text"])
with training_args.main_process_first(desc="dataset map tokenization"): # 防多卡重复
    tokenized_datasets = ds.map(
        tokenize_function, # callback
        pass
    )

def group_texts(examples): # local scope 局部作用域
    return do_something(examples)
with training_args.main_process_first(desc="文本分块"):
    lm_datasets = tokenized_datasets.map(
        group_texts,
        pass
    ) 
```  
  

关键在于`ds.map(tokenize_function)`，注意，没有加括号！

如果加了括号 tokenize_function(某数据)：这叫 **“调用函数”**。意思是：“我亲自把数据塞给你，你现在立刻给我干活！”

没加括号 tokenize_function：这叫 **“传递函数对象”**。意思是：“我把这个‘工人’交给你了。”

在这里，ds.map 就是那个包工头。
当你执行 ds.map(tokenize_function) 时，你是在对包工头说：“这是我雇的翻译工（tokenize_function），怎么干活我都教好他了，你负责给他安排工作。”

ds.map 在底层的真实运作过程是这样的（伪代码）：
```python
def map(self, function_you_passed_in):
    new_dataset = []
    # 1. 包工头自己去仓库（ds）里搬一箱数据
    for batch_data in self.get_batches(): 
        # 2. 包工头强行把这箱数据塞进你给的工人手里（赋值给 examples）
        processed_data = function_you_passed_in(batch_data) 
        new_dataset.append(processed_data)
    return new_dataset
```
结论：不是你没有给 examples 赋值，而是 ds.map 在后台替你自动完成了赋值和循环。它每次从你的 32GB 数据集里挖出一大块，偷偷传给了 tokenize_function 里的 examples。

# 字典推导式
## 代码行解释

你问的这行代码：
```python
concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
```

### 核心功能：
这行代码的作用是**将批处理（batch）中的多个文本样本拼接成一个连续的长序列**。

### 详细解释：

1. **上下文**：这段代码出现在 `group_texts` 函数中，该函数用于语言模型预训练的数据预处理。

2. **输入**：`examples` 是一个字典，包含一个批次的多个样本。例如：
   ```python
   examples = {
     'input_ids': [[1, 2, 3], [4, 5, 6], [7, 8, 9]],  # 3个样本，每样本3个token
     'attention_mask': [[1, 1, 1], [1, 1, 1], [1, 1, 1]]
   }
   ```

3. **处理过程**：
   - `examples.keys()`：获取所有字段名（如 `['input_ids', 'attention_mask']`）
   - 对每个字段：
     - `examples[k]`：获取该字段的所有样本值
     - `*examples[k]`：解包列表，将 `[[1,2,3], [4,5,6], [7,8,9]]` 变成三个独立参数
     - `chain(...)`：使用 `itertools.chain` 将多个列表连接成一个连续的迭代器
     - `list(...)`：将迭代器转换为列表

4. **输出**：处理后得到：
   ```python
   concatenated_examples = {
     'input_ids': [1, 2, 3, 4, 5, 6, 7, 8, 9],  # 所有样本拼接成一个长序列
     'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]
   }
   ```


### 类比理解：
想象你有3个短句子：
1. "我喜欢"
2. "吃苹果"
3. "和香蕉"

拼接后变成："我喜欢吃苹果和香蕉"，然后按照固定长度（如5个字符）分块：
- "我喜欢吃"
- "苹果和香"
- "蕉"

这样模型就能学习到跨句子的语言模式。



# 列表推导式
## 代码解释：

```python
result = {
    k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
    for k, t in concatenated_examples.items()
}
```

### 1. __`concatenated_examples.items()`__：

- `concatenated_examples` 是我们之前创建的字典，包含拼接后的数据

- `.items()` 返回字典的键值对，例如：

  ```python
  [('input_ids', [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]), 
   ('attention_mask', [1, 1, 1, 1, 1, 1, 1, 1, 1, 1])]
  ```

### 2. __`for k, t in concatenated_examples.items()`__：

- 这是一个循环，遍历字典的每个键值对
- `k`：键（key），例如 `'input_ids'`、`'attention_mask'`
- `t`：值（value），例如 `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`
- __`t` 就是拼接后的长列表__，包含该字段的所有值

### 3. __`t[i : i + block_size]`__：

- `t` 是拼接后的长列表
- `i` 是起始索引
- `i + block_size` 是结束索引
- 这是一&#x4E2A;__&#x5207;片操作__，从列表 `t` 中取出从索引 `i` 到 `i+block_size` 的子列表

### 4. __`[t[i : i + block_size] for i in range(0, total_length, block_size)]`__：

- 这是一&#x4E2A;__&#x5217;表推导式__

- `range(0, total_length, block_size)`：生成起始索引序列

  - 例如：如果 `total_length=10`, `block_size=3`
  - 则 `range(0, 10, 3)` → `[0, 3, 6, 9]`

- 对每个起始索引 `i`，取出 `t[i:i+3]` 的切片

- 结果是一个列表的列表

### 5. __完整示例__：

假设：

```python
concatenated_examples = {
    'input_ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],  # t = 这个列表
    'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}
total_length = 10
block_size = 3
```

处理过程：

1. 第一个键值对：`k='input_ids'`, `t=[1,2,3,4,5,6,7,8,9,10]`

   - `range(0, 10, 3)` → `[0, 3, 6, 9]`
   - `i=0`: `t[0:3]` → `[1, 2, 3]`
   - `i=3`: `t[3:6]` → `[4, 5, 6]`
   - `i=6`: `t[6:9]` → `[7, 8, 9]`
   - `i=9`: `t[9:12]` → `[10]`（注意：最后一个块可能小于block_size）
   - 结果：`[[1,2,3], [4,5,6], [7,8,9], [10]]`

2. 第二个键值对：`k='attention_mask'`, `t=[1,1,1,1,1,1,1,1,1,1]`

   - 同样切片：`[[1,1,1], [1,1,1], [1,1,1], [1]]`

最终 `result`：

```python
{
    'input_ids': [[1,2,3], [4,5,6], [7,8,9], [10]],
    'attention_mask': [[1,1,1], [1,1,1], [1,1,1], [1]]
}
```



#  Pretrain VS SFT
| 阶段                                                    | 数据类型     | 训练目标             | 任务形式             | Loss 计算范围             |
| ----------------------------------------------------- | -------- | ---------------- | ---------------- | --------------------- |
| **Pretrain**                                          | 海量无监督文本  | 学习语言规律、语义结构和世界知识 | CLM（预测下一个 token） | **整个文本**              |
| **SFT** (Supervised Fine-Tuning)                      | 指令-回答数据对(Prompt-Response) | 让模型学会遵循用户指令完成任务  | CLM（根据指令生成回答）    | **只计算回答部分**           |
| **RLHF** (Reinforcement Learning from Human Feedback) | 人类偏好数据   | 让模型输出更符合人类偏好     | 强化学习             | 不直接用 token-level loss |

Pretrain 会对全部 text 进行 loss 计算，要求模型对整个文本实现建模预测；而 SFT 仅对输出进行 loss 计算，不计算指令部分的 loss。